# Dataset Regression Testing

This notebook shows how to use golden datasets to prevent regressions, run A/B tests, and integrate with Langfuse Experiments.

**What you'll learn:**

1. **The Offline Loop** — From detecting issues to preventing them
2. **Basic Dataset Evaluation** — `@langfuse_dataset` with `@fe.correctness`
3. **Langfuse Experiments** — How runs appear in the Experiments comparison UI
4. **Metadata Flattening** — Use custom dataset fields in your test functions
5. **Input-Only Datasets** — When there's no expected output
6. **A/B Testing** — Compare agent versions side-by-side
7. **Promoting Traces to Datasets** — Turn production failures into test cases
8. **CI/CD Gates** — Block deployments on quality regressions

Every cell is **runnable**. Make sure you've completed the [Overview notebook](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/overview.ipynb) first.

---

## Setup

In [ ]:
!pip install -q fasteval-core fasteval-langfuse

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
    os.environ["LANGFUSE_HOST"] = userdata.get("LANGFUSE_HOST")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Keys loaded from Colab Secrets.")
except (ImportError, Exception):
    # os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
    # os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
    # os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"
    # os.environ["OPENAI_API_KEY"] = "sk-..."
    print("Using environment variables.")

In [ ]:
import fasteval as fe
from fasteval_langfuse import langfuse_dataset, langfuse_traces, configure_langfuse, LangfuseConfig

configure_langfuse(LangfuseConfig(auto_push_scores=True))
print("fasteval + langfuse ready.")

---

## 1. The Offline Loop

While trace evaluation (online) catches issues in production, dataset evaluation (offline) **prevents them from shipping**.

The lifecycle:

```
1. DETECT       Trace evaluation finds a failure in production
                "Bot hallucinated about refund policy"
      │
      ▼
2. PROMOTE      Turn that trace into a dataset item
                input: "What's your refund policy?"
                expected_output: "We offer 30-day refunds for..."
      │
      ▼
3. FIX          Update the agent/prompt/retrieval
      │
      ▼
4. VERIFY       Run dataset evaluation to confirm the fix works
                AND didn't break other test cases (regression)
      │
      ▼
5. DEPLOY       Dataset tests pass in CI → green light to ship
```

Over time, your golden dataset grows with every production failure, creating an increasingly robust test suite.

---

## 2. Creating a Sample Dataset

Before we can evaluate, we need a dataset in Langfuse. Let's create a small one programmatically.

> **Colab Note:** You can also create datasets via the Langfuse UI or CSV upload. This code creates one for demonstration purposes. If you already have a dataset, skip to Section 3.

In [ ]:
from langfuse import Langfuse

langfuse = Langfuse()

DATASET_NAME = "fasteval-cookbook-demo"

# Create dataset (idempotent — safe to run multiple times)
langfuse.create_dataset(
    name=DATASET_NAME,
    description="Demo dataset for fasteval Langfuse cookbook",
)
print(f"Dataset '{DATASET_NAME}' ready.")

In [ ]:
# Add sample items
items = [
    {
        "input": "What is your refund policy?",
        "expected_output": "We offer a 30-day money-back guarantee on all plans. Contact support to initiate a refund.",
        "metadata": {"category": "billing", "difficulty": "easy"},
    },
    {
        "input": "How do I reset my password?",
        "expected_output": "Go to Settings > Security > Change Password. You can also use the 'Forgot Password' link on the login page.",
        "metadata": {"category": "account", "difficulty": "easy"},
    },
    {
        "input": "Can I use the API with Python 3.8?",
        "expected_output": "Yes, our Python SDK supports Python 3.8 and above. Install with: pip install our-sdk",
        "metadata": {"category": "technical", "difficulty": "medium"},
    },
    {
        "input": "What's the difference between the Pro and Enterprise plans?",
        "expected_output": "Pro includes 10k API calls/month and email support. Enterprise includes unlimited calls, SSO, SLA, and dedicated support.",
        "metadata": {"category": "billing", "difficulty": "medium"},
    },
    {
        "input": "How do I configure webhook notifications for failed deployments?",
        "expected_output": "Go to Settings > Integrations > Webhooks. Add a URL and select 'deployment.failed' as the event trigger.",
        "metadata": {"category": "technical", "difficulty": "hard"},
    },
]

for item in items:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=item["input"],
        expected_output=item["expected_output"],
        metadata=item["metadata"],
    )

print(f"Added {len(items)} items to '{DATASET_NAME}'.")
for i, item in enumerate(items, 1):
    print(f"  {i}. [{item['metadata']['category']}] {item['input'][:50]}")

---

## 3. Basic Dataset Evaluation

Use `@langfuse_dataset` to iterate through a dataset. The decorator calls your function **once per item**, passing `input` and `expected_output` from the dataset.

> **Colab Note:** The decorator handles fetching, iteration, and score reporting. You just write the evaluation logic.

In [ ]:
# Mock agent — replace with your real agent
def my_agent_v1(query: str) -> str:
    """Baseline agent (simulated)."""
    responses = {
        "refund": "We offer a 30-day money-back guarantee. Email support@example.com.",
        "password": "Go to Settings > Security > Change Password.",
        "python": "Yes, we support Python 3.8+. Install with pip install our-sdk.",
        "pro": "Pro has 10k calls/month. Enterprise has unlimited calls and SSO.",
        "webhook": "Go to Settings > Integrations > Webhooks and add your URL.",
    }
    for key, response in responses.items():
        if key in query.lower():
            return response
    return f"I can help with that. Here's information about: {query}"


print("Agent v1 ready.")
print(f"  Sample: {my_agent_v1('What is your refund policy?')}")

In [ ]:
@fe.correctness(threshold=0.8)
@langfuse_dataset(name=DATASET_NAME)
def test_agent_v1(input, expected_output):
    """Test baseline agent against golden dataset."""
    response = my_agent_v1(input)
    print(f"  Q: {input[:50]}")
    print(f"  A: {response[:80]}")
    print(f"  Expected: {expected_output[:80]}")
    print()
    fe.score(response, expected_output, input=input)


print(f"Evaluating agent_v1 against '{DATASET_NAME}'...\n")
test_agent_v1()

---

## 4. Langfuse Experiments Integration

When you run `@langfuse_dataset`, the integration creates a **Dataset Run** in Langfuse:

1. A unique **run name** is generated (e.g., `fasteval-test_agent_v1-2026-03-05T10:00:00`)
2. Each item execution creates a **Trace** linked to the dataset item
3. Scores (`fasteval_correctness`, etc.) are pushed to each trace
4. Results appear in the **Langfuse Experiments UI**

This means:
- You can see results for each dataset item in Langfuse
- Multiple runs are compared side-by-side in the Experiments tab
- Score trends over time are tracked automatically

**To view:** Go to Langfuse > Datasets > Select your dataset > Experiments tab.

---

## 5. Metadata Flattening

If your dataset items have metadata (e.g., `{"difficulty": "hard", "category": "billing"}`), these are **flattened** into keyword arguments. Just declare them in your function signature.

This is useful for:
- Conditional evaluation logic (e.g., lower threshold for hard questions)
- Logging and debugging
- Filtering results by category

In [ ]:
@fe.correctness(threshold=0.7)
@langfuse_dataset(name=DATASET_NAME)
def test_with_metadata(input, expected_output, difficulty, category):
    """Use metadata fields to customize evaluation logic."""
    response = my_agent_v1(input)

    print(f"  [{category}/{difficulty}] {input[:40]}")

    if difficulty == "hard":
        print(f"    (hard question — being more lenient)")

    fe.score(response, expected_output, input=input)


print(f"Evaluating with metadata...\n")
test_with_metadata()

---

## 6. Input-Only Datasets

Sometimes your dataset has only inputs (no expected output). This is useful for:
- Smoke testing ("does the agent crash?")
- Judge-only metrics ("is the response relevant to the input?")
- Toxicity/safety checks

Just omit `expected_output` from your function signature.

In [ ]:
@fe.relevance(threshold=0.7)
@langfuse_dataset(name=DATASET_NAME)
def test_relevance_only(input):
    """Check relevance without expected output."""
    response = my_agent_v1(input)
    print(f"  Q: {input[:50]}")
    print(f"  A: {response[:80]}")
    print()
    fe.score(response, input=input)


print("Relevance-only evaluation (no expected output needed):\n")
test_relevance_only()

---

## 7. A/B Testing

Compare two versions of your agent on the same dataset. Each version generates a separate run in Langfuse Experiments, enabling side-by-side comparison.

### Step 1: Define Agent v2

In [ ]:
def my_agent_v2(query: str) -> str:
    """Improved agent (simulated — more detailed responses)."""
    responses = {
        "refund": "We offer a full 30-day money-back guarantee on all plans, no questions asked. To initiate a refund, contact support@example.com or use the self-service option in Settings > Billing > Request Refund.",
        "password": "To reset your password: 1) Go to Settings > Security > Change Password, or 2) Click 'Forgot Password' on the login page and follow the email instructions. Passwords must be 12+ characters.",
        "python": "Yes! Our Python SDK supports Python 3.8 and above. Install with: pip install our-sdk. See our quickstart guide at docs.example.com/python for setup instructions.",
        "pro": "Pro plan ($49/mo): 10k API calls, email support, basic analytics. Enterprise (custom pricing): unlimited calls, SSO/SAML, 99.9% SLA, dedicated account manager, and priority support.",
        "webhook": "To configure webhook notifications: 1) Go to Settings > Integrations > Webhooks, 2) Click 'Add Webhook', 3) Enter your URL, 4) Select 'deployment.failed' from the events list, 5) Test the webhook, 6) Save.",
    }
    for key, response in responses.items():
        if key in query.lower():
            return response
    return f"I'd be happy to help with that. Could you provide more details about: {query}"


print("Agent v2 ready (more detailed responses).")
print(f"  Sample: {my_agent_v2('What is your refund policy?')[:100]}...")

### Step 2: Run Both Versions on the Same Dataset

In [ ]:
# Run 1: Baseline (v1)
@fe.correctness(threshold=0.85)
@fe.relevance(threshold=0.8)
@langfuse_dataset(name=DATASET_NAME)
def test_baseline_v1(input, expected_output):
    """Baseline agent for A/B comparison."""
    response = my_agent_v1(input)
    print(f"  [v1] {input[:40]} → {response[:50]}")
    fe.score(response, expected_output, input=input)


print("=" * 60)
print("A/B Test: Running Agent v1 (baseline)...")
print("=" * 60)
test_baseline_v1()

In [ ]:
# Run 2: New version (v2)
@fe.correctness(threshold=0.85)
@fe.relevance(threshold=0.8)
@langfuse_dataset(name=DATASET_NAME)
def test_candidate_v2(input, expected_output):
    """New agent version for A/B comparison."""
    response = my_agent_v2(input)
    print(f"  [v2] {input[:40]} → {response[:50]}")
    fe.score(response, expected_output, input=input)


print("=" * 60)
print("A/B Test: Running Agent v2 (candidate)...")
print("=" * 60)
test_candidate_v2()

### Step 3: Compare in Langfuse

After running both tests:

1. Go to **Langfuse > Datasets > `fasteval-cookbook-demo`**
2. Click the **Experiments** tab
3. You'll see two runs side by side:
   - `fasteval-test_baseline_v1-<timestamp>`
   - `fasteval-test_candidate_v2-<timestamp>`
4. Compare scores per item and overall averages
5. Click individual items to see the judge's reasoning

This makes it easy to answer: **"Did v2 actually improve quality?"**

---

## 8. Promoting Traces to Datasets

The most powerful pattern: turn production failures into permanent test cases. This bridges online evaluation (traces) with offline evaluation (datasets).

### Pattern: Find failures → Add to golden set

> **Colab Note:** This pattern uses `@langfuse_traces` (from the [Basic Trace Evaluation](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/basic-trace-evaluation.ipynb) notebook) to find low-quality traces, then the Langfuse SDK to promote them.

In [ ]:
from fasteval_langfuse.sampling import ScoreBasedSamplingStrategy

promoted_traces = []


@langfuse_traces(
    time_range="last_7d",
    sampling=ScoreBasedSamplingStrategy(
        score_name="user_rating",
        low_score_threshold=3.0,
        low_score_rate=1.0,
        high_score_rate=0.0,
    ),
    limit=5,
)
def find_failures_to_promote(trace_id, input, output, metadata):
    """Find low-rated traces to promote to the golden dataset."""
    promoted_traces.append({
        "trace_id": trace_id,
        "input": input,
        "output": output,
    })
    print(f"  Found: {trace_id[:12]} — {str(input)[:50]}")


print("Searching for low-rated traces to promote...\n")
find_failures_to_promote()
print(f"\nFound {len(promoted_traces)} traces to review.")

In [ ]:
# Promote traces to dataset (with human-written expected output)
#
# In practice, a human expert would review the trace and write
# the correct expected_output before adding it to the dataset.

for trace in promoted_traces:
    print(f"Promoting trace {trace['trace_id'][:12]}:")
    print(f"  Input:  {str(trace['input'])[:60]}")
    print(f"  Output: {str(trace['output'])[:60]} (this was wrong)")
    print()

    # Uncomment to actually create the dataset item:
    # langfuse.create_dataset_item(
    #     dataset_name=DATASET_NAME,
    #     input=trace["input"],
    #     expected_output="<HUMAN EXPERT: Write the correct answer here>",
    #     metadata={
    #         "source_trace_id": trace["trace_id"],
    #         "promoted_from": "production_failure",
    #     },
    # )

print(f"Review complete. {len(promoted_traces)} traces ready for promotion.")
print("Uncomment the langfuse.create_dataset_item() call to add them.")

---

## 9. Dataset Versioning and Organization

### Versioning

Use the `version` parameter to pin evaluations to a specific dataset version. This ensures reproducibility.

```python
@langfuse_dataset(name="qa-golden-set", version="v3")
def test_pinned_version(input, expected_output):
    ...
```

### Dataset Folders

Langfuse supports organizing datasets with `/` in names:

```python
langfuse.create_dataset(name="evaluation/support/billing")
langfuse.create_dataset(name="evaluation/support/technical")
langfuse.create_dataset(name="evaluation/rag/qa")
```

This is useful when your team manages multiple evaluation domains.

---

## 10. CI/CD Gates

Use dataset tests as deployment gates. If quality drops below your threshold, the build fails.

### pytest Test File

```python
# tests/test_regression.py
import fasteval as fe
from fasteval_langfuse import langfuse_dataset


@fe.correctness(threshold=0.85)
@fe.relevance(threshold=0.8)
@langfuse_dataset(name="qa-golden-set", version="v3")
def test_agent_regression(input, expected_output):
    from my_app import agent
    response = agent(input)
    fe.score(response, expected_output, input=input)
```

### GitHub Actions Workflow

```yaml
# .github/workflows/deploy.yml
name: Deploy

on:
  push:
    branches: [main]

jobs:
  regression-test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install -r requirements.txt
      - name: Run regression tests
        env:
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: pytest tests/test_regression.py -v --tb=short

  deploy:
    needs: regression-test   # Only deploy if tests pass
    runs-on: ubuntu-latest
    steps:
      - run: echo "Deploying..."
```

If any metric score falls below the threshold on any dataset item (or the aggregate), pytest fails, and the deploy job is blocked.

---

## 11. Cleanup (Optional)

Remove the demo dataset if you don't want it lingering in your Langfuse project.

In [ ]:
# Uncomment to delete the demo dataset:
# langfuse.client.datasets.delete(DATASET_NAME)
# print(f"Deleted dataset '{DATASET_NAME}'.")

print(f"Demo dataset '{DATASET_NAME}' is still in Langfuse.")
print("Uncomment the line above to delete it.")

---

## Summary

In this notebook, you learned:

- How `@langfuse_dataset` evaluates every item in a dataset
- How results appear in Langfuse Experiments for comparison
- How to use metadata flattening for conditional evaluation
- How to run A/B tests between agent versions
- How to promote production failures into golden test cases
- How to use dataset tests as CI/CD deployment gates

### The Full Cookbook

| Notebook | What it covers |
|----------|---------------|
| [Overview](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/overview.ipynb) | Installation, configuration, evaluation loop |
| [Basic Trace Evaluation](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/basic-trace-evaluation.ipynb) | Fetch and evaluate production traces |
| [Sampling Strategies](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/sampling-strategies.ipynb) | Cost optimization with smart sampling |
| **This notebook** | Golden datasets, A/B testing, CI/CD gates |

For the full API reference, see the [fasteval-langfuse plugin docs](https://fasteval.dev/docs/plugins/langfuse/overview).